# Regularized logistic regression (Esther)

## 0. Importations

In [6]:
from helpers import *
from implementations import *
from preprocessing import *
from validation import *

## 1. Data loading & preprocessing

In [4]:
from pathlib import Path  # BEGIN: ed8c6549bwf9
project_dir = Path.cwd()  # Updated to use Path.cwd()
data_dir = project_dir / "data" / "dataset"
output_dir = project_dir / "outputs"
output_dir.mkdir(exist_ok=True)

x_train, x_test, y_train, _, test_ids = load_csv_data(data_dir)
names = load_feature_names(data_dir)

In [5]:
tx_train, tx_test = clean_and_standardize(x_train, x_test, names)
y01 = (y_train == 1).astype(float)   # {-1,1} -> {0,1} pour la régression logistique

## 2. Split train & validation set

In [9]:
tx_tr, tx_va, y_tr, y_va = split_train_validation(y_train, tx_train)   # labels en {-1, 1}
y_tr01 = (y_tr == 1).astype(float) 

## 3. First training (verify the loss decrease)

In [10]:
max_iters, gamma = 500, 0.5
w0 = np.zeros(tx_train.shape[1])

w, loss = reg_logistic_regression(y_tr01, tx_tr, 1e-4, w0, max_iters, gamma)
print("loss train :", loss, "| loss validation :", logistic_loss((y_va == 1).astype(float), tx_va, w))

scores = tx_va @ w
t = best_threshold(y_va, scores)
f1 = f1_score(y_va, np.where(scores >= t, 1, -1))
print(f"F1 validation = {f1:.3f} (seuil {t:.2f})")

loss train : 0.22663116131505265 | loss validation : 0.2276518050479557
F1 validation = 0.416 (seuil -1.38)


## 4. Lambda tuning

In [11]:
results = []
for lambda_ in [0, 1e-5, 1e-4, 1e-3, 1e-2]:
    w, _ = reg_logistic_regression(y_tr01, tx_tr, lambda_, w0, max_iters, gamma)
    scores = tx_va @ w
    t = best_threshold(y_va, scores)
    f1 = f1_score(y_va, np.where(scores >= t, 1, -1))
    results.append((f1, lambda_, t))
    print(f"lambda = {lambda_:g}  ->  F1 = {f1:.3f}  (seuil {t:.2f})")

best_f1, best_lambda, best_t = max(results)
print("meilleur :", best_lambda, best_t, best_f1)

lambda = 0  ->  F1 = 0.416  (seuil -1.38)
lambda = 1e-05  ->  F1 = 0.416  (seuil -1.45)
lambda = 0.0001  ->  F1 = 0.416  (seuil -1.38)
lambda = 0.001  ->  F1 = 0.417  (seuil -1.46)
lambda = 0.01  ->  F1 = 0.413  (seuil -1.12)
meilleur : 0.001 -1.4589850059858345 0.4168194597237501


## 5. Final training & Submission

In [12]:
w, loss = reg_logistic_regression(y01, tx_train, best_lambda, w0, max_iters, gamma)
print("loss train :", loss, "| loss validation :", logistic_loss((y_va == 1).astype(float), tx_va, w))

scores = tx_va @ w
t = best_threshold(y_va, scores)
f1 = f1_score(y_va, np.where(scores >= t, 1, -1))
print(f"F1 validation = {f1:.3f} (seuil {t:.2f})")

loss train : 0.22771585642698902 | loss validation : 0.22819828457209074
F1 validation = 0.418 (seuil -1.46)


In [14]:
y_pred = np.where(tx_test @ w >= best_t, 1, -1)
create_csv_submission(test_ids, y_pred, output_dir / "submission_reg_logistic.csv")
print("prédits positifs :", (y_pred == 1).mean())

prédits positifs : 0.1593084595763355
